In [3]:
import glob
import os
import sqlite3
import pandas as pd

In [4]:
#Nom de la base de données
db_name = "risque_database.db"

#Supprimer l'ancienne base si elle existe pour repartir de zéro
if os.path.exists(db_name):
    os.remove(db_name)

#Connexion unique
conn = sqlite3.connect(db_name)
cursor = conn.cursor()

In [5]:
#Activer les clés étrangères
cursor.execute("PRAGMA foreign_keys = ON;")

CREATE TABLE IF NOT EXISTS nomtable

In [10]:
cursor.executescript("""
CREATE TABLE IF NOT EXISTS NAF5 (
    id_naf5 INTEGER PRIMARY KEY AUTOINCREMENT,
    code_NAF5 TEXT NOT NULL UNIQUE,
    libelle_NAF5 TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS Risque_lib (
    id_risque INTEGER PRIMARY KEY AUTOINCREMENT,
    risque TEXT NOT NULL UNIQUE
);

CREATE TABLE IF NOT EXISTS Cause_lib (
    id_cause INTEGER PRIMARY KEY AUTOINCREMENT,
    libelle_cause_TR TEXT NOT NULL UNIQUE
);

CREATE TABLE IF NOT EXISTS Partie_lib (
    id_partie INTEGER PRIMARY KEY AUTOINCREMENT,
    partie_corps TEXT NOT NULL UNIQUE
);

CREATE TABLE IF NOT EXISTS Region_lib (
    id_region INTEGER PRIMARY KEY AUTOINCREMENT,
    libelle_region_admin TEXT NOT NULL UNIQUE
);

CREATE TABLE IF NOT EXISTS Activite (
    id_activite INTEGER PRIMARY KEY AUTOINCREMENT,
    annee INTEGER NOT NULL,
    nb_jours_arret_travail REAL,
    id_naf5 INTEGER NOT NULL REFERENCES NAF5(id_naf5),
    id_risque INTEGER NOT NULL REFERENCES Risque_lib(id_risque),
    UNIQUE (annee, id_naf5, id_risque)
);

CREATE TABLE IF NOT EXISTS Cause (
    id_cause_fait INTEGER PRIMARY KEY AUTOINCREMENT,
    annee INTEGER NOT NULL,
    id_naf5 INTEGER NOT NULL REFERENCES NAF5(id_naf5),
    id_cause INTEGER NOT NULL REFERENCES Cause_lib(id_cause),
    UNIQUE (annee, id_naf5, id_cause)
);

CREATE TABLE IF NOT EXISTS HF (
    id_hf INTEGER PRIMARY KEY AUTOINCREMENT,
    annee INTEGER NOT NULL,
    sexe TEXT NOT NULL CHECK (sexe IN ('Hommes', 'Femmes')),
    id_naf5 INTEGER NOT NULL REFERENCES NAF5(id_naf5),
    UNIQUE (annee, id_naf5, sexe)
);

CREATE TABLE IF NOT EXISTS PartieCorps (
    id_partie_fait INTEGER PRIMARY KEY AUTOINCREMENT,
    annee INTEGER NOT NULL,
    id_naf5 INTEGER NOT NULL REFERENCES NAF5(id_naf5),
    id_partie INTEGER NOT NULL REFERENCES Partie_lib(id_partie),
    UNIQUE (annee, id_naf5, id_partie)
);

CREATE TABLE IF NOT EXISTS SinistresRegion (
    id_sinistre INTEGER PRIMARY KEY AUTOINCREMENT,
    annee INTEGER NOT NULL,
    nb_salaries INTEGER,
    nb_sinistres_int INTEGER,
    id_region INTEGER NOT NULL REFERENCES Region_lib(id_region),
    id_risque INTEGER NOT NULL REFERENCES Risque_lib(id_risque),
    UNIQUE (annee, id_region, id_risque)
);

CREATE TABLE IF NOT EXISTS Depenses (
    id_depense INTEGER PRIMARY KEY AUTOINCREMENT,
    annee INTEGER NOT NULL,
    montant_depenses INTEGER,
    poste_depenses TEXT NOT NULL,
    id_risque INTEGER NOT NULL REFERENCES Risque_lib(id_risque),
    UNIQUE (annee, id_risque, poste_depenses)
);
""")
conn.commit()
conn.commit();

## Insertion des données depuis les CSV

In [11]:
sep = ";"

# 1) Tables de référence (lib) : insérées telles quelles
naf5 = pd.read_csv("df_df_NAF5_lib.csv", sep=sep)
naf5.to_sql("NAF5", conn, if_exists="append", index=False)

cause_lib = pd.read_csv("df_cause_lib.csv", sep=sep)
cause_lib.to_sql("Cause_lib", conn, if_exists="append", index=False)

partie_lib = pd.read_csv("df_parti_lib.csv", sep=sep)
partie_lib.to_sql("Partie_lib", conn, if_exists="append", index=False)

region_lib = pd.read_csv("df_region_lib.csv", sep=sep)
region_lib.to_sql("Region_lib", conn, if_exists="append", index=False)

# Fichiers "faits" (nécessaires pour construire Risque_lib et pour l'étape suivante)
activite = pd.read_csv("df_activite-nettoye.csv", sep=sep)
cause = pd.read_csv("df_cause-nettoye.csv", sep=sep)
hf = pd.read_csv("df_hf-nettoye.csv", sep=sep)
partie = pd.read_csv("df_partie-nettoye.csv", sep=sep)
region = pd.read_csv("df_region-nettoye.csv", sep=sep)
depense = pd.read_csv("df_depense-nettoye.csv", sep=sep)

# Risque_lib : pas de fichier dédié -> valeurs uniques des fichiers qui contiennent "risque"
risques = pd.concat([activite["risque"], region["risque"], depense["risque"]]).drop_duplicates()
pd.DataFrame({"risque": risques}).to_sql("Risque_lib", conn, if_exists="append", index=False)

conn.commit()

In [12]:
# 2) Récupérer les id générés par SQLite pour pouvoir faire les jointures
id_naf5 = pd.read_sql("SELECT id_naf5, code_NAF5 FROM NAF5", conn)
id_risque = pd.read_sql("SELECT id_risque, risque FROM Risque_lib", conn)
id_cause = pd.read_sql("SELECT id_cause, libelle_cause_TR FROM Cause_lib", conn)
id_partie = pd.read_sql("SELECT id_partie, partie_corps FROM Partie_lib", conn)
id_region = pd.read_sql("SELECT id_region, libelle_region_admin FROM Region_lib", conn)

In [13]:
# 3) Tables de faits : on remplace les libellés par les id via merge, puis on insère

df = activite.merge(id_naf5, on="code_NAF5").merge(id_risque, on="risque")
df = df[["annee", "nb_jours_arret_travail", "id_naf5", "id_risque"]]
df.to_sql("Activite", conn, if_exists="append", index=False)

df = cause.merge(id_naf5, on="code_NAF5").merge(id_cause, on="libelle_cause_TR")
df = df[["annee", "id_naf5", "id_cause"]]
df.to_sql("Cause", conn, if_exists="append", index=False)

df = hf.merge(id_naf5, on="code_NAF5")
df = df[["annee", "sexe", "id_naf5"]]
df.to_sql("HF", conn, if_exists="append", index=False)

df = partie.merge(id_naf5, on="code_NAF5").merge(id_partie, on="partie_corps")
df = df[["annee", "id_naf5", "id_partie"]]
df.to_sql("PartieCorps", conn, if_exists="append", index=False)

df = region.merge(id_region, on="libelle_region_admin").merge(id_risque, on="risque")
df = df[["annee", "nb_salaries", "nb_sinistres_int", "id_region", "id_risque"]]
df.to_sql("SinistresRegion", conn, if_exists="append", index=False)

df = depense.merge(id_risque, on="risque")
df = df[["annee", "montant_depenses", "poste_depenses", "id_risque"]]
df.to_sql("Depenses", conn, if_exists="append", index=False)

conn.commit()